In [ ]:
# Load necessary libraries
import shap
import os
import pickle
import joblib
import pandas as pd
import pyreadr

1. Create SHAP explainer from the PRIME model 

In [ ]:
# directories
working_dir = "model_evaluation"
model_dir = "PRIME_GM12878_model_1.0.sav"

os.chdir(working_dir)

# load the trained model
with open(model_dir, 'rb') as file:
    model = pickle.load(file)

# SHAP explainer
explainer = shap.TreeExplainer(model)
joblib.dump(explainer, "shap_explainer_PRIME_GM12878_model_1.0.pkl")

2. Prepare the selected profiles to be ready for generating SHAP profile

In [ ]:

from python.prepare_shap_data import prepare_shap_data

profile_dir = "K562_endres_profiles/profiles_subtnorm/"

# Extracts and processes test profiles based on a filename filter.
# Ensures that the final result retains the correct indices that match the profile.

# Save POSITIVE samples
test_X_pos = prepare_shap_data(profile_dir,
                               file_filter="K562_C",
                               output_path=os.path.join(working_dir, "K562_C_endres_v2"),
                               sample_size=None)

# test_X_pos = prepare_shap_data(profile_dir,
#                                file_filter="K562_N",
#                                output_path=os.path.join(working_dir, "K562_N_endres_v2"),
#                                sample_size=None)

3. Compute SHAP values from POSITIVE profiles

In [ ]:

# make sure the explainer is existing and loaded correctly
# also as profiles
print(type(explainer))
profiles = test_X_pos["test_X"]
print(f"Loaded filtered profiles: {profiles.shape}")

# compute SHAP values for the profiles
shap_values = explainer(profiles)

# save SHAP values to Pickle
outname = "K562_C_endres_v2"
#outname = "K562_N_endres_v2"
shap_pkl_path = "SHAP_value_"+outname+".pkl"
with open(shap_pkl_path, "wb") as file:
    pickle.dump(shap_values, file)
print(f"SHAP values saved to {shap_pkl_path}")

4. Gather all data for plotting

In [ ]:
# Combine BED files of all chromosomes into one file for each prefix (e.g., K562_N, K562_C)

# Function to combine BED files based on prefix before "_chr"
# Takes two arguments: input directory and output directory
combine_bed_files() {
  local input_dir="$1"
  local output_dir="$2"

  # Ensure the output directory exists
  mkdir -p "$output_dir"

  # Find all unique prefixes before "_chr"
  prefixes=$(ls "$input_dir"/*.csv | sed -E 's/(.*)(_chr[^_]+).*/\1/' | sort -u)

  # For each unique prefix, concatenate all matching files
  for prefix in $prefixes; do
    # Find all files that match the prefix (with any chr)
    matching_files=$(ls "$input_dir"/$(basename "$prefix")_chr*.csv)

    # Combine those files into one file in the output directory
    combined_file="$output_dir/$(basename "$prefix")_combined.csv"

    # Handle headers: Include header from the first file, skip for others
    head -n 1 $(echo $matching_files | cut -d ' ' -f1) > "$combined_file"  # Extract header from the first file
    for file in $matching_files; do
      tail -n +2 "$file" >> "$combined_file"  # Skip header (start from line 2)
    done

    echo "Combined files into $combined_file"
  done

}

combine_bed_files /K562_endres_profiles/profiles_subtnorm/K562_C /K562_endres_profiles/profiles_subtnorm
combine_bed_files /K562_endres_profiles/profiles_subtnorm/K562_N /K562_endres_profiles/profiles_subtnorm

combine_bed_files /K562_endres_profiles/profiles/K562_C /K562_endres_profiles/profiles
combine_bed_files /K562_endres_profiles/profiles/K562_N /K562_endres_profiles/profiles

In [ ]:
# set up paths and parameters

subtnorm_profile_path = profile_dir + "K562_C_combined.csv"
count_dir = "K562_endres_profiles/profiles/"
count_profile_path = count_dir + "K562_C_combined.csv"

manual_colnames = range(-200, 201)

In [ ]:
# SHAP_values
shap_df = pd.DataFrame(shap_values.values[:, :, 1], columns=manual_colnames)
print(shap_df.head())
print(shap_df.shape)

# SHAP final indices
shap_index_df = test_X_pos["final_indices"]
print(shap_index_df.head())
print(shap_index_df.shape)
shap_df.index = shap_index_df[0].values
print(shap_df.head())

In [ ]:
# profile subtnorm
subtnorm_df = pd.read_csv(subtnorm_profile_path, header=0, index_col="rownames")
subtnorm_df.index.name = None
subtnorm_df.columns = manual_colnames
print(subtnorm_df.head())
print(subtnorm_df.shape)

In [ ]:
# profile count
count_df = pd.read_csv(count_profile_path, header=0, index_col="rownames")
count_df.index.name = None
print(count_df.head())
print(count_df.shape)

# Split the DataFrame into two based on column prefixes
plus_count_df = count_df.loc[:, count_df.columns.str.startswith("Plus")]
plus_count_df.columns = manual_colnames
minus_count_df = count_df.loc[:, count_df.columns.str.startswith("Minus")]
minus_count_df.columns = manual_colnames
print(plus_count_df.head())
print(plus_count_df.shape)
print(minus_count_df.head())
print(minus_count_df.shape)

In [ ]:
# Check if all DataFrames have the same index, column names (order matters), and shape
dataframes = [shap_df, subtnorm_df, plus_count_df, minus_count_df]
index_check = all(dataframes[0].index.equals(df.index) for df in dataframes)
columns_check = all(dataframes[0].columns.equals(df.columns) for df in dataframes)
shape_check = all(dataframes[0].shape == df.shape for df in dataframes)

print(f"All DataFrames have the same index: {index_check}")
print(f"All DataFrames have the same column names: {columns_check}")
print(f"All DataFrames have the same shape: {shape_check}")

In [ ]:
# Save each DataFrame as an RDS file
pyreadr.write_rds(outname + "_shap_value_df.rds", shap_df)
pyreadr.write_rds(outname + "_profile_subtnorm_df.rds", subtnorm_df)
pyreadr.write_rds(outname + "_profile_count_plus_df.rds", plus_count_df)
pyreadr.write_rds(outname + "_profile_count_minus_df.rds", minus_count_df)

index_df = pd.DataFrame(minus_count_df.index)
index_df.columns = ["rownames"]
index_df.index.name = None
print(index_df.head())
pyreadr.write_rds(outname + "_index.rds", index_df)

print("All DataFrames saved as .rds files.")

5. Read in data to R and preapare the final object for visualization

In [ ]:
# Load necessary libraries
library(GenomicRanges)
library(readr)

In [ ]:
head(shap_df)
head(index_df)

rownames(shap_df) <- index_df$rownames
rownames(subtnorm_df) <- index_df$rownames
rownames(plus_count_df) <- index_df$rownames
rownames(minus_count_df) <- index_df$rownames

parts <- strsplit(index_df$rownames, "[:;-]")
genomic_df <- do.call(rbind, parts)
genomic_df <- as.data.frame(genomic_df)
colnames(genomic_df) <- c("seqnames", "start", "end", "strand")

# convert start and end to numeric
genomic_df$start <- as.numeric(genomic_df$start)
genomic_df$end <- as.numeric(genomic_df$end)

# convert to GRanges object
gr <- GRanges(
  seqnames = genomic_df$seqnames,
  ranges = IRanges(start = genomic_df$start, end = genomic_df$end),
  strand = genomic_df$strand
)

ls <- list()
ls$shap_value <- shap_df
ls$profile_subtnorm <- subtnorm_df
ls$profile_count_plus <- plus_count_df
ls$profile_count_minus <- minus_count_df
ls$gr <- gr

saveRDS(ls, file = paste0(outname, "_shap-subtnorm-plus-minus-gr.rds"))